## 01. Request

In [47]:
import uuid, json
from app.core.time import now_kst_str
from app.langgraph.common.schema import ChatRequest

print(ChatRequest.model_fields)

{'thread_id': FieldInfo(annotation=Union[str, NoneType], required=False, default=None), 'message': FieldInfo(annotation=str, required=True, metadata=[MinLen(min_length=1)])}


In [48]:
message = ChatRequest(
    message="내 이름이 뭐라고 ?",
    thread_id="bbb023f0"
    )

thread_id = message.thread_id or str(uuid.uuid4())[:8]
# thread_id가 없는경우 자동으로 생성

state = {
    "thread_id": thread_id,
    "request": message,
}


In [49]:
state

{'thread_id': 'bbb023f0',
 'request': ChatRequest(thread_id='bbb023f0', message='내 이름이 뭐라고 ?')}

## 02. Build Messages 

In [50]:
from app.langgraph.common.state import ChatState
from app.langgraph.common.chat_memory import ChatMemory
from app.core.redis import get_redis_client
from app.core.config import CHAT_SYSTEM_PROMPT_PATH

In [51]:
# System Prompt
with open(CHAT_SYSTEM_PROMPT_PATH, "r", encoding="utf-8") as f:
    system_prompt = f.read()

print(system_prompt)

You are a chatbot assistant for a log analysis platform.
Please answer questions concisely and accurately in Korean.


In [52]:
# Redis에서 채팅 내역 불러오기
redis_client = get_redis_client()
chat_memory = ChatMemory(redis_client)

history = chat_memory.get(state["thread_id"])

print(history)

[{'role': 'user', 'content': '내 이름은 황재식이다.'}, {'role': 'assistant', 'content': '안녕하세요, 황재식 님. 무엇을 도와드릴까요?'}]


In [54]:
# 최종 프롬프트
state["messages"] = [
    {"role": "system", "content": system_prompt},
    *history,
    {"role": "user", "content": state["request"].message},
]

state

{'thread_id': 'bbb023f0',
 'request': ChatRequest(thread_id='bbb023f0', message='내 이름이 뭐라고 ?'),
 'messages': [{'role': 'system',
   'content': 'You are a chatbot assistant for a log analysis platform.\nPlease answer questions concisely and accurately in Korean.'},
  {'role': 'user', 'content': '내 이름은 황재식이다.'},
  {'role': 'assistant', 'content': '안녕하세요, 황재식 님. 무엇을 도와드릴까요?'},
  {'role': 'user', 'content': '내 이름이 뭐라고 ?'}]}

## 03. Call LLM

In [55]:
import httpx

In [56]:
MODEL_NAME = "gpt-oss-20b"
VLLM_BASE_URL2 = "http://10.122.100.173:8000/v1/chat/completions"

In [57]:
# 비동기 HTTP 클라이언트 전역 생성 ( 재사용을 통해 커넥션 풀 활용 및 성능 최적화 )
client = httpx.AsyncClient(
    timeout=httpx.Timeout(
        connect=5.0,   # TCP 연결 수립 최대 대기 시간
        read=120.0,    # 응답 바디 수신 최대 대기 시간 (LLM 추론 고려)
        write=30.0,    # 요청 바디 전송 최대 시간
        pool=5.0       # 커넥션 풀에서 대기 가능한 최대 시간
    ),
    limits=httpx.Limits(
        max_connections=50,          # 동시에 열 수 있는 최대 TCP 연결 수
        max_keepalive_connections=20 # Keep-Alive 유지 연결 수
    ),
)

In [58]:
payload = {
    "model": MODEL_NAME,
    "messages": state["messages"],
    "temperature": 0.0,
    "max_tokens": 8196,
}

resp = await client.post(VLLM_BASE_URL2, json=payload)
resp.raise_for_status()

state["llm_raw"] = resp.json()


In [59]:
state

{'thread_id': 'bbb023f0',
 'request': ChatRequest(thread_id='bbb023f0', message='내 이름이 뭐라고 ?'),
 'messages': [{'role': 'system',
   'content': 'You are a chatbot assistant for a log analysis platform.\nPlease answer questions concisely and accurately in Korean.'},
  {'role': 'user', 'content': '내 이름은 황재식이다.'},
  {'role': 'assistant', 'content': '안녕하세요, 황재식 님. 무엇을 도와드릴까요?'},
  {'role': 'user', 'content': '내 이름이 뭐라고 ?'}],
 'llm_raw': {'id': 'chatcmpl-b590d4402e5a7ac4',
  'object': 'chat.completion',
  'created': 1777255985,
  'model': 'gpt-oss-20b',
  'choices': [{'index': 0,
    'message': {'role': 'assistant',
     'content': '당신의 이름은 **황재식**입니다.',
     'refusal': None,
     'annotations': None,
     'audio': None,
     'function_call': None,
     'tool_calls': [],
     'reasoning': 'User says: "내 이름이 뭐라고 ?" They want to know their name. They previously said "내 이름은 황재식이다." So answer: "황재식" in Korean.'},
    'logprobs': None,
    'finish_reason': 'stop',
    'stop_reason': None,
    'to

## 04. Extract Answer

In [60]:
data = state["llm_raw"]
state["answer"] = data["choices"][0]["message"]["content"]

In [61]:
state

{'thread_id': 'bbb023f0',
 'request': ChatRequest(thread_id='bbb023f0', message='내 이름이 뭐라고 ?'),
 'messages': [{'role': 'system',
   'content': 'You are a chatbot assistant for a log analysis platform.\nPlease answer questions concisely and accurately in Korean.'},
  {'role': 'user', 'content': '내 이름은 황재식이다.'},
  {'role': 'assistant', 'content': '안녕하세요, 황재식 님. 무엇을 도와드릴까요?'},
  {'role': 'user', 'content': '내 이름이 뭐라고 ?'}],
 'llm_raw': {'id': 'chatcmpl-b590d4402e5a7ac4',
  'object': 'chat.completion',
  'created': 1777255985,
  'model': 'gpt-oss-20b',
  'choices': [{'index': 0,
    'message': {'role': 'assistant',
     'content': '당신의 이름은 **황재식**입니다.',
     'refusal': None,
     'annotations': None,
     'audio': None,
     'function_call': None,
     'tool_calls': [],
     'reasoning': 'User says: "내 이름이 뭐라고 ?" They want to know their name. They previously said "내 이름은 황재식이다." So answer: "황재식" in Korean.'},
    'logprobs': None,
    'finish_reason': 'stop',
    'stop_reason': None,
    'to

In [62]:
redis_client = get_redis_client()
chat_memory = ChatMemory(redis_client)

# 채팅 내역 저장
chat_memory.append_turn(
    thread_id=state["thread_id"],
    user_message=state["request"].message,
    assistant_message=state["answer"],
)

state["history_count"] = len(chat_memory.get(state["thread_id"]))

In [63]:
print(chat_memory.get(state["thread_id"]))

[{'role': 'user', 'content': '내 이름은 황재식이다.'}, {'role': 'assistant', 'content': '안녕하세요, 황재식 님. 무엇을 도와드릴까요?'}, {'role': 'user', 'content': '내 이름이 뭐라고 ?'}, {'role': 'assistant', 'content': '당신의 이름은 **황재식**입니다.'}]


## 05. Return

In [64]:
from app.langgraph.common.schema import ChatResponse

In [65]:
answer = state.get("answer")
history_count = state.get("history_count")

ChatResponse(
        thread_id=thread_id,
        answer=answer,
        history_count=history_count or 0,
    )

ChatResponse(thread_id='bbb023f0', answer='당신의 이름은 **황재식**입니다.', history_count=4)